# Exploration

In [1]:
# load model
import torch
from transformers import AutoProcessor, AutoModel

# I modified one file in tranformers to make it work:
# https://github.com/huggingface/transformers/pull/38637/files

model_clip = AutoModel.from_pretrained(
    "openai/clip-vit-base-patch32", torch_dtype=torch.float32
)  # be careful with alternative attention implementations

processor = AutoProcessor.from_pretrained(
    "openai/clip-vit-base-patch32",
)


/Users/przemek/anaconda3/envs/executorch-2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0612 14:03:08.905000 36784 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [2]:
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
labels = ["a photo of a cat", "a photo of a dog", "a photo of a car"]

inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)

outputs = model_clip(**inputs)

In [3]:
for child in model_clip.named_children():
    print(child, end="\n###############################\n")

('text_model', CLIPTextTransformer(
  (embeddings): CLIPTextEmbeddings(
    (token_embedding): Embedding(49408, 512)
    (position_embedding): Embedding(77, 512)
  )
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (layer_norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (final

# Text model

In [4]:
class TextModel(torch.nn.Module):
    def __init__(self, text_model, text_projection):
        super().__init__()
        self.text_model =text_model 
        self.text_projection = text_projection


    def forward(self, input_ids, attention_mask = None):
        # add special tokens - we couldn't do it with tokenizers at a time
        # hardcoded based on models tokenizer
        input_ids = torch.cat((torch.tensor([[49406]], dtype=input_ids.dtype), input_ids, torch.tensor([[49407]], dtype=input_ids.dtype)), 1)
        if (attention_mask is not None):
            attention_mask = torch.cat((torch.tensor([[1]], dtype=attention_mask.dtype), attention_mask, torch.tensor([[1]], dtype=attention_mask.dtype)), 1)
        
        output = self.text_model(input_ids, attention_mask)
        output = output.pooler_output
        output = self.text_projection(output)
        output = torch.flatten(output)
        return output



text_model = TextModel(model_clip.text_model, model_clip.text_projection)

In [5]:
text_input = "bike"
input_ids = processor(text=text_input, return_tensors="pt", padding="max_length", truncation = "longest_first", max_length=74, add_special_tokens=False)
outputs1 = text_model(input_ids["input_ids"])

text_input = "car"
input_ids = processor(text=text_input, return_tensors="pt", padding="max_length", truncation = "longest_first", max_length=74,add_special_tokens=False)
outputs2 = text_model(input_ids["input_ids"])

text_input = "dog jumping on a ferrari"
input_ids = processor(text=text_input, return_tensors="pt", padding="max_length", truncation = "longest_first", max_length=74,add_special_tokens=False)
outputs3 = text_model(input_ids["input_ids"], input_ids['attention_mask'])

In [6]:
cos = torch.nn.CosineSimilarity(dim=0, eps=1e-6)
cos(outputs1, outputs2), cos(outputs1, outputs3), cos(outputs2, outputs3)

(tensor(0.8558, grad_fn=<SumBackward1>),
 tensor(0.7501, grad_fn=<SumBackward1>),
 tensor(0.7590, grad_fn=<SumBackward1>))

In [7]:
import torch
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower
from torch.export import export, Dim

input_ids2 = (torch.randint(0, 20000, (1,20), dtype=torch.int64), torch.randint(0, 2, (1,20), dtype=torch.int64))

sequence_length = Dim("sequence_length", min=0, max=74)
# Specify that the first dimension of each input is that batch size
dynamic_shapes = {"input_ids": {1: sequence_length}, "attention_mask": {1: sequence_length}}


exported_program = export(text_model, input_ids2, dynamic_shapes=dynamic_shapes)
executorch_program = to_edge_transform_and_lower(
    exported_program,
    partitioner = [XnnpackPartitioner()]
).to_executorch()

with open("clip_text_model_NEW.pte", "wb") as file:
    file.write(executorch_program.buffer)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/przemek/anaconda3/envs/executorch-2/lib/python3.10/site-packages/torch/export/_unlift.py:81: UserWarning: Attempted to insert a get_attr Node with no underlying reference in the owning GraphModule! Call GraphModule.add_submodule to add the necessary submodule, GraphModule.add_parameter to add the necessary Parameter, or nn.Module.register_buffer to add the necessary buffer
  getattr_node = gm.graph.get_attr(lifted_node)
/Users/przemek/anaconda3/envs/executorch-2/lib/python3.10/site-packages/torch/fx/graph.py:1772: UserWarning: Node lifted_tensor_0 target lifted_tensor_0 lifted_tensor_0 of  does not reference an nn.Module, nn.Parameter, or buffer, which is what 

In [8]:
from executorch.runtime import Runtime

runtime = Runtime.get()

input_tensor = torch.randint(0, 20000, (1,1), dtype=torch.int64)
input_tensor2 = torch.randint(0, 2, (1,1), dtype=torch.int64)
program = runtime.load_program("clip_text_model.pte")
method = program.load_method("forward")
outputs = method.execute([input_tensor, input_tensor2])

print(outputs[0].shape, outputs[0].dtype, len(outputs), type(outputs))

[program.cpp:136] InternalConsistency verification requested but not available


torch.Size([512]) torch.float32 1 <class 'list'>


# Now vision model

In [12]:
class VisionModel(torch.nn.Module):
    def __init__(self, vision_model, visual_projection):
        super().__init__()
        self.vision_model = vision_model 
        self.visual_projection = visual_projection

    def forward(self, pixel_values):
        output = self.vision_model(pixel_values)
        output = output.pooler_output
        output = self.visual_projection(output)
        output = torch.flatten(output)
        return output

vision_model = VisionModel(model_clip.vision_model, model_clip.visual_projection)

In [13]:
# cat
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
inputs = processor(images=image, return_tensors="pt", padding=True)
outputs1_vis = vision_model(inputs["pixel_values"])

# bike
url = "https://farm2.staticflickr.com/1084/1148759967_49e79af6a5_z.jpg"
image = Image.open(requests.get(url, stream=True).raw)
inputs = processor(images=image, return_tensors="pt", padding=True)
outputs2_vis = vision_model(inputs["pixel_values"])

# dog
url = "https://farm8.staticflickr.com/7218/7255487026_288b791eba_z.jpg"
image = Image.open(requests.get(url, stream=True).raw)
inputs = processor(images=image, return_tensors="pt", padding=True)
outputs3_vis = vision_model(inputs["pixel_values"])

In [14]:
# outputs1 = bike, outputs3 = dog jumping on a ferrari
cos(outputs1, outputs3_vis), cos(outputs3, outputs3_vis)

(tensor(0.1535, grad_fn=<SumBackward1>),
 tensor(0.2746, grad_fn=<SumBackward1>))

In [15]:
cos(outputs1_vis, outputs3_vis), cos(outputs2_vis, outputs3_vis)

(tensor(0.6237, grad_fn=<SumBackward1>),
 tensor(0.4879, grad_fn=<SumBackward1>))

In [16]:
input_ids3 = (torch.randn(1, 3, 224, 224, dtype=torch.float32),)

exported_program = export(vision_model, input_ids3)
executorch_program = to_edge_transform_and_lower(
    exported_program,
    partitioner = [XnnpackPartitioner()]
).to_executorch()

with open("clip_vision_model_NEW.pte", "wb") as file:
    file.write(executorch_program.buffer)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [18]:
runtime = Runtime.get()

input_tensor = torch.randn(1, 3, 224, 224, dtype=torch.float32)
program = runtime.load_program("clip_vision_model.pte")
method = program.load_method("forward")
outputs = method.execute([input_tensor])

print(outputs[0].shape)

[program.cpp:136] InternalConsistency verification requested but not available


torch.Size([512])


# Final test in executorch from python

In [19]:
torch.set_printoptions(precision=4, sci_mode=False)

In [20]:
text = "two cute kittens"
input_tensor = processor(text=text, return_tensors="pt", truncation = "longest_first", max_length=76)

program = runtime.load_program("clip_text_model.pte")
method = program.load_method("forward")
output_text = method.execute([input_tensor["input_ids"], input_tensor["attention_mask"]])[0]

[program.cpp:136] InternalConsistency verification requested but not availableprogram.cpp:136] InternalConsistency verification requested but not available


In [21]:
text = "angry man"
input_tensor = processor(text=text, return_tensors="pt", truncation = "longest_first", max_length=76)

output_text2 = method.execute([input_tensor["input_ids"], input_tensor["attention_mask"]])[0]

In [25]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
input_tensor = processor(images=image, return_tensors="pt", padding=True)

program = runtime.load_program("clip_vision_model.pte")
method = program.load_method("forward")
output_vision = method.execute([input_tensor["pixel_values"]])[0]

[program.cpp:136] InternalConsistency verification requested but not availableprogram.cpp:136] InternalConsistency verification requested but not available


In [26]:
cos = torch.nn.CosineSimilarity(dim=0, eps=1e-6)
cos(output_vision, output_text), cos(output_vision, output_text2)

(tensor(0.2719), tensor(0.1936))